# Task 1 — Dataset Identification & Analysis

**Capstone: Fitness-Adapt — Personalised Real-Time Exercise Assessment**

This notebook identifies suitable datasets, loads the project metadata, analyses label distributions, and prepares data structures for downstream pose-estimation and modelling.

**Primary reference:** Parmar, P., Gharat, A., & Rhodin, H. (2022). *Domain Knowledge-Informed Self-Supervised Representations for Workout Form Assessment*. ECCV 2022. [arXiv 2202.14019](https://arxiv.org/abs/2202.14019)

## 1.1 Imports and Setup

In [ ]:
import json
import os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

ROOT = Path(".")
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

print("Setup complete.")

## 1.2 Dataset Survey

We need datasets that contain **two types of labels**:
1. **Exercise type** (e.g., squat, lunge, push-up)
2. **Quality / error label** indicating how correctly the exercise was performed

### Candidate Datasets

| Dataset | Exercises | Quality Labels | Public | Notes |
|---------|-----------|---------------|--------|-------|
| **Fitness-AQA** (Parmar et al., ECCV 2022) | BackSquat, BarbellRow, OverheadPress | Per-rep error annotations (knees inward, knees forward, etc.) | Request-access | **Primary dataset for this project** — metadata already present |
| **FLEX** (Yin et al., 2025) | 20 weight-loaded exercises | Expert annotations in Fitness Knowledge Graph | GitHub | Large-scale, multi-modal (RGB + 3D pose + sEMG) |
| **M3GYM** (Xu et al., CVPR 2025) | 51+ activities (gym, Pilates, yoga) | Action completeness scores | Request-access | Multi-view, multi-person |
| **InfiniteRep** (Infinity AI) | 10 exercises (squat, push-up, etc.) | Synthetic — joint angles & bounding boxes | GitHub | 1 000 synthetic videos, good for pre-training |
| **NTU RGB+D 120** | 120 actions (incl. exercises) | Action class only (no quality) | Request-access | Useful for exercise-type classification pre-training |
| **Fit3D** (Fieraru et al., 2021) | Fitness exercises | 3D pose but no quality labels | Request-access | Useful for 3D pose baselines |

### Decision

**Fitness-AQA** is the primary dataset because:
- It directly supports the capstone's research paper (arXiv 2202.14019)
- It has both exercise-type and per-rep quality/error annotations
- Metadata (splits, error labels) is already in this repository
- Expert trainers annotated specific biomechanical errors (knees inward, knees forward)

**Supplementary:** InfiniteRep provides synthetic data with joint angles for augmenting training.  
**Future work:** FLEX dataset could be integrated for multi-exercise coverage.

## 1.3 Load Project Metadata

In [ ]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

train_keys = load_json(ROOT / "train_keys.json")
val_keys   = load_json(ROOT / "val_keys.json")
test_keys  = load_json(ROOT / "test_keys.json")

error_knees_inward  = load_json(ROOT / "error_knees_inward.json")
error_knees_forward = load_json(ROOT / "error_knees_forward.json")
traj_nan            = load_json(ROOT / "traj_nan.json")

print(f"Train clips : {len(train_keys)}")
print(f"Val clips   : {len(val_keys)}")
print(f"Test clips  : {len(test_keys)}")
print(f"Total clips : {len(train_keys) + len(val_keys) + len(test_keys)}")
print(f"\nKnees-inward annotations : {len(error_knees_inward)} clips")
print(f"Knees-forward annotations: {len(error_knees_forward)} clips")
print(f"Clips with NaN trajectories: {len(traj_nan)}")

## 1.4 Clip ID Structure Analysis

Each clip ID follows the format `VIDEO_ID_CLIP_NUMBER` (e.g., `50530_1`).  
Let's understand the distribution of videos and clips.

In [ ]:
all_keys = train_keys + val_keys + test_keys

def parse_clip_id(clip_id):
    parts = clip_id.rsplit("_", 1)
    return parts[0], int(parts[1])

video_ids = [parse_clip_id(k)[0] for k in all_keys]
clip_nums = [parse_clip_id(k)[1] for k in all_keys]

unique_videos = set(video_ids)
clips_per_video = Counter(video_ids)

print(f"Unique videos: {len(unique_videos)}")
print(f"Total clips  : {len(all_keys)}")
print(f"\nClips per video — min: {min(clips_per_video.values())}, "
      f"max: {max(clips_per_video.values())}, "
      f"mean: {np.mean(list(clips_per_video.values())):.1f}")
print(f"Clip number range: {min(clip_nums)} – {max(clip_nums)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cpv = list(clips_per_video.values())
axes[0].hist(cpv, bins=range(1, max(cpv) + 2), edgecolor="black", alpha=0.8)
axes[0].set_xlabel("Clips per Video")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Clips per Video")

split_counts = [len(train_keys), len(val_keys), len(test_keys)]
split_labels = ["Train", "Validation", "Test"]
colors = ["#4C72B0", "#DD8452", "#55A868"]
bars = axes[1].bar(split_labels, split_counts, color=colors, edgecolor="black")
for bar, count in zip(bars, split_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(count), ha="center", fontweight="bold")
axes[1].set_ylabel("Number of Clips")
axes[1].set_title("Train / Validation / Test Split")

plt.tight_layout()
plt.savefig("data/split_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: data/split_distribution.png")

## 1.5 Error Annotation Analysis

The Fitness-AQA dataset annotates two common squat errors with temporal intervals:
- **Knees Inward** (valgus collapse) — knees cave toward each other
- **Knees Forward** — knees travel excessively past the toes

Empty list `[]` = no error present in that clip; non-empty = list of `[start_sec, end_sec]` intervals.

In [ ]:
def analyse_error_annotations(error_dict, error_name):
    """Analyse a single error annotation dictionary."""
    total = len(error_dict)
    has_error = {k: v for k, v in error_dict.items() if v}
    no_error  = {k: v for k, v in error_dict.items() if not v}

    durations = []
    for clip_id, intervals in has_error.items():
        for interval in intervals:
            if len(interval) == 2:
                durations.append(interval[1] - interval[0])

    intervals_per_clip = [len(v) for v in has_error.values()]

    print(f"=== {error_name} ===")
    print(f"  Total annotated clips : {total}")
    print(f"  Clips WITH error      : {len(has_error)} ({100*len(has_error)/total:.1f}%)")
    print(f"  Clips WITHOUT error   : {len(no_error)} ({100*len(no_error)/total:.1f}%)")
    if durations:
        print(f"  Error intervals total : {sum(intervals_per_clip)}")
        print(f"  Duration (sec) — mean: {np.mean(durations):.2f}, "
              f"std: {np.std(durations):.2f}, "
              f"min: {np.min(durations):.2f}, max: {np.max(durations):.2f}")
    print()
    return has_error, durations

ki_errors, ki_durations = analyse_error_annotations(error_knees_inward, "Knees Inward")
kf_errors, kf_durations = analyse_error_annotations(error_knees_forward, "Knees Forward")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Error presence ---
ki_counts = [len(ki_errors), len(error_knees_inward) - len(ki_errors)]
kf_counts = [len(kf_errors), len(error_knees_forward) - len(kf_errors)]

axes[0, 0].pie(ki_counts, labels=["Has Error", "No Error"], autopct="%1.1f%%",
               colors=["#E07B54", "#72B5A4"], startangle=90)
axes[0, 0].set_title("Knees Inward — Error Presence")

axes[0, 1].pie(kf_counts, labels=["Has Error", "No Error"], autopct="%1.1f%%",
               colors=["#E07B54", "#72B5A4"], startangle=90)
axes[0, 1].set_title("Knees Forward — Error Presence")

# --- Duration distributions ---
if ki_durations:
    axes[1, 0].hist(ki_durations, bins=30, edgecolor="black", alpha=0.8, color="#E07B54")
    axes[1, 0].axvline(np.mean(ki_durations), color="red", linestyle="--",
                        label=f"Mean = {np.mean(ki_durations):.2f}s")
    axes[1, 0].set_xlabel("Error Duration (seconds)")
    axes[1, 0].set_ylabel("Count")
    axes[1, 0].set_title("Knees Inward — Error Duration Distribution")
    axes[1, 0].legend()

if kf_durations:
    axes[1, 1].hist(kf_durations, bins=30, edgecolor="black", alpha=0.8, color="#4C72B0")
    axes[1, 1].axvline(np.mean(kf_durations), color="red", linestyle="--",
                        label=f"Mean = {np.mean(kf_durations):.2f}s")
    axes[1, 1].set_xlabel("Error Duration (seconds)")
    axes[1, 1].set_ylabel("Count")
    axes[1, 1].set_title("Knees Forward — Error Duration Distribution")
    axes[1, 1].legend()

plt.tight_layout()
plt.savefig("data/error_annotation_analysis.png", bbox_inches="tight")
plt.show()
print("Saved: data/error_annotation_analysis.png")

## 1.6 Combined Quality Labels

To support the quality scorer (Task 6), we derive a continuous quality score per clip.  
A clip with **no errors** gets score 1.0; clips with errors get lower scores proportional to error duration.

In [ ]:
all_annotated_clips = set(error_knees_inward.keys()) | set(error_knees_forward.keys())
print(f"Total annotated clips (union of both error types): {len(all_annotated_clips)}")

def compute_error_duration(intervals):
    """Total error duration in seconds from a list of [start, end] intervals."""
    return sum(end - start for start, end in intervals if len([start, end]) == 2)

records = []
for clip_id in sorted(all_annotated_clips):
    ki_intervals = error_knees_inward.get(clip_id, [])
    kf_intervals = error_knees_forward.get(clip_id, [])

    ki_dur = compute_error_duration(ki_intervals)
    kf_dur = compute_error_duration(kf_intervals)

    has_ki = len(ki_intervals) > 0
    has_kf = len(kf_intervals) > 0

    split = "train" if clip_id in train_keys else (
            "val"   if clip_id in val_keys else (
            "test"  if clip_id in test_keys else "unknown"))

    video_id, clip_num = parse_clip_id(clip_id)

    records.append({
        "clip_id": clip_id,
        "video_id": video_id,
        "clip_num": clip_num,
        "split": split,
        "has_knees_inward": has_ki,
        "has_knees_forward": has_kf,
        "knees_inward_duration": ki_dur,
        "knees_forward_duration": kf_dur,
        "total_error_duration": ki_dur + kf_dur,
        "error_count": int(has_ki) + int(has_kf),
    })

df = pd.DataFrame(records)
print(f"\nDataFrame shape: {df.shape}")
df.head(10)

In [ ]:
MAX_ERROR_DUR = df["total_error_duration"].quantile(0.95)
df["quality_score"] = 1.0 - np.clip(df["total_error_duration"] / MAX_ERROR_DUR, 0, 1)

print("Quality score statistics:")
print(df["quality_score"].describe())
print(f"\nClips with perfect score (1.0): {(df['quality_score'] == 1.0).sum()}")
print(f"Clips with errors (< 1.0)     : {(df['quality_score'] < 1.0).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Quality score distribution
axes[0].hist(df["quality_score"], bins=40, edgecolor="black", alpha=0.8, color="#55A868")
axes[0].set_xlabel("Quality Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Quality Score Distribution (0 = worst, 1 = perfect)")

# Error co-occurrence
error_combos = df.groupby(["has_knees_inward", "has_knees_forward"]).size().reset_index(name="count")
labels_combo = []
for _, row in error_combos.iterrows():
    ki = "KI" if row["has_knees_inward"] else "no-KI"
    kf = "KF" if row["has_knees_forward"] else "no-KF"
    labels_combo.append(f"{ki} + {kf}")
axes[1].bar(labels_combo, error_combos["count"], color=["#72B5A4", "#E07B54", "#4C72B0", "#C44E52"],
            edgecolor="black")
axes[1].set_ylabel("Count")
axes[1].set_title("Error Co-occurrence")
axes[1].tick_params(axis="x", rotation=15)

# Quality score by split
for split, color in zip(["train", "val", "test"], ["#4C72B0", "#DD8452", "#55A868"]):
    subset = df[df["split"] == split]["quality_score"]
    axes[2].hist(subset, bins=30, alpha=0.6, label=f"{split} (n={len(subset)})", color=color)
axes[2].set_xlabel("Quality Score")
axes[2].set_ylabel("Count")
axes[2].set_title("Quality Score by Split")
axes[2].legend()

plt.tight_layout()
plt.savefig("data/quality_score_analysis.png", bbox_inches="tight")
plt.show()
print("Saved: data/quality_score_analysis.png")

## 1.7 NaN Trajectory Analysis

In [ ]:
nan_clip_ids = [f.replace(".json", "") for f in traj_nan]
print(f"Clips with NaN trajectories ({len(nan_clip_ids)}):")
for cid in nan_clip_ids:
    in_train = cid in train_keys
    in_val   = cid in val_keys
    in_test  = cid in test_keys
    split = "train" if in_train else ("val" if in_val else ("test" if in_test else "none"))
    print(f"  {cid:>15s}  split={split}")

print(f"\nThese clips should be excluded or handled with imputation during preprocessing.")

## 1.8 Create Master Dataset CSV

Save the combined annotation DataFrame for downstream use in other notebooks.

In [ ]:
nan_set = set(nan_clip_ids)
df["has_nan_trajectory"] = df["clip_id"].isin(nan_set)
print(f"Clips flagged with NaN trajectory: {df['has_nan_trajectory'].sum()}")

df["exercise_type"] = "BackSquat"

output_path = DATA_DIR / "dataset_master.csv"
df.to_csv(output_path, index=False)
print(f"\nSaved master dataset: {output_path}")
print(f"Shape: {df.shape}")
print(f"\nColumn summary:")
print(df.dtypes)

In [ ]:
split_summary = df.groupby("split").agg(
    n_clips=("clip_id", "count"),
    n_with_ki=("has_knees_inward", "sum"),
    n_with_kf=("has_knees_forward", "sum"),
    mean_quality=("quality_score", "mean"),
    std_quality=("quality_score", "std"),
    n_nan_traj=("has_nan_trajectory", "sum"),
).round(3)

print("=== Split Summary ===")
split_summary

## 1.9 Summary & Next Steps

### Key Findings

1. **Primary dataset**: Fitness-AQA (from arXiv 2202.14019) — BackSquat videos with expert error annotations
2. **Dataset size**: The repository metadata covers ~1 000+ clips across train/val/test splits
3. **Error types**: Two biomechanical errors annotated — *knees inward* and *knees forward* — with temporal intervals
4. **Quality score**: Derived continuous quality score (0–1) from error durations suitable for regression
5. **Data quality**: 19 clips have NaN trajectories and need special handling

### Supplementary Datasets Identified
- **InfiniteRep**: 1 000 synthetic exercise videos with joint angles (10 exercise types)
- **FLEX**: 7 500+ multi-view recordings of 20 exercises with expert annotations
- **M3GYM**: 82 gym sessions with 2D/3D keypoints and action completeness scores

### Next Steps
→ **Task 2** (Notebook `02_pose_estimation.ipynb`): Apply pose estimation models (MediaPipe, YOLOv11-Pose, ViTPose) to extract body keypoints and compare their performance.